# Textract

Runs Amazon Textract once on the sample scans, note photos, and email screenshots, and saves its full answer, one JSON per page, in `data/textract/`. The evaluation notebook and the tests read these saved answers, so they need no AWS.

Each page is one Textract call (a few seconds, charged per page). A page already saved is skipped, so rerunning costs nothing; to read a page again, delete its JSON. Needs AWS credentials on this machine. The photos aren't in the repository, so the photo cells only run where they are on disk.

## Setup

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, "../src")
from intake import timekeeper_table
from textract import page_images, run_textract

In [2]:
SHEETS_PATH = Path("../data/dev_scans/2399_001.pdf")       # 4 handwritten time sheets
NOTES_DIR = Path("../data/dev_photos")                    # phone photos of notes (JPEG or PNG)
SCREENSHOTS_DIR = Path("../data/dev_typed/screenshots")   # screenshots of typed emails (PNG)
OUTSIDE_PHOTOS = Path("../data/held_out/photos")          # notes from outside writers
TEXTRACT_DIR = Path("../data/textract/dev")
OUTSIDE_TEXTRACT = Path("../data/textract/held_out")
TIMEKEEPERS_PATH = Path("../data/timekeepers.csv")
COLUMN_NOTES = ["mon_fri_sample.jpg", "note_07.JPG"]      # notes with drawn columns, also read as a sheet

REGION = "us-west-2"
DPI = 200        # PDF pages are rendered at this resolution before Textract reads them

In [3]:
def images(folder):
    if not folder.exists():
        return []
    found = sorted(p for p in folder.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".heic"})
    assert not [p for p in found if p.suffix.lower() == ".heic"], "Convert HEIC photos to JPEG first"
    return found


photos, outside = images(NOTES_DIR), images(OUTSIDE_PHOTOS)
screenshots = sorted(SCREENSHOTS_DIR.glob("*.png"))
timekeepers = timekeeper_table(pd.read_csv(TIMEKEEPERS_PATH))
assert SHEETS_PATH.exists() and timekeepers["tk_id"].is_unique
print(f"{len(page_images(SHEETS_PATH, DPI))} sheet pages | {len(photos)} note photos | {len(outside)} outside writers' "
      f"photos | {len(screenshots)} email screenshots")

4 sheet pages | 19 note photos | 10 outside writers' photos | 3 email screenshots


## Run Textract

Scans are read in `sheet` mode (tables and form fields); photos and screenshots in `note` mode (lines only, the cheapest). Notes with drawn columns, and the outside writers' photos, are read both ways.

In [4]:
sheet_files = run_textract(SHEETS_PATH, TEXTRACT_DIR, "sheet", REGION, DPI)
note_files = [f for p in photos for f in run_textract(p, TEXTRACT_DIR, "note", REGION)]
column_files = [f for p in photos if p.name in COLUMN_NOTES for f in run_textract(p, TEXTRACT_DIR, "sheet", REGION)]
screenshot_files = [f for p in screenshots for f in run_textract(p, TEXTRACT_DIR, "note", REGION)]
outside_files = [f for p in outside for mode in ("note", "sheet") for f in run_textract(p, OUTSIDE_TEXTRACT, mode, REGION)]
print(f"{len(sheet_files)} sheet pages, {len(note_files)} note photos, {len(column_files)} column notes as sheets, "
      f"{len(screenshot_files)} screenshots, {len(outside_files)} pages of outside writers' photos")

4 sheet pages, 19 note photos, 2 column notes as sheets, 3 screenshots, 20 pages of outside writers' photos
